In [1]:

import sys
import platform

print("¡Proyecto EDA UTEC iniciado correctamente!")
print("Versión de Python:", sys.version.split()[0])
print("Sistema operativo:", platform.system())
print("Entorno:", sys.executable)


¡Proyecto EDA UTEC iniciado correctamente!
Versión de Python: 3.12.10
Sistema operativo: Windows
Entorno: c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Scripts\python.exe


In [1]:

import numpy as np
import pandas as pd
import matplotlib
import pyarrow
import duckdb
import psutil

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("PyArrow:", pyarrow.__version__)
print("DuckDB:", duckdb.__version__)
print("psutil:", psutil.__version__)

print("\n¡Librerías instaladas correctamente!")


NumPy: 2.5.4
Pandas: 3.1.0
Matplotlib: 3.11.2
PyArrow: 26.0.0
DuckDB: 2.0.0
psutil: 7.2.2

¡Librerías instaladas correctamente!


## Verificar el archivo desde Jupyter

In [2]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_PATH = PROJECT_DIR / "data" / "raw" / "2019-Oct.csv"

print("Ruta:", DATA_PATH)
print("Archivo encontrado:", DATA_PATH.exists())

if DATA_PATH.exists():
    size_gb = DATA_PATH.stat().st_size / (1024**3)
    print(f"Tamaño: {size_gb:.2f} GiB")


Ruta: c:\Users\bryan\Proyecto_EDA_UTEC\data\raw\2019-Oct.csv
Archivo encontrado: True
Tamaño: 5.28 GiB


## Exploración inicial del dataset REES46

In [3]:

import pandas as pd
import time

# Tamaño inicial de la muestra
N_SAMPLE = 100_000

# Medir el tiempo de lectura
inicio = time.perf_counter()

df_sample = pd.read_csv(
    DATA_PATH,
    nrows=N_SAMPLE
)

tiempo = time.perf_counter() - inicio

print(f"Registros cargados: {len(df_sample):,}")
print(f"Tiempo de lectura: {tiempo:.3f} segundos")
print(f"Columnas: {df_sample.shape[1]}")
print(
    f"Memoria utilizada: "
    f"{df_sample.memory_usage(deep=True).sum() / 1024**2:.2f} MiB"
)

display(df_sample.head(10))


Registros cargados: 100,000
Tiempo de lectura: 0.237 segundos
Columnas: 9
Memoria utilizada: 14.85 MiB


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
5,2019-10-01 00:00:05 UTC,view,1480613,2053013561092866779,computers.desktop,pulser,908.62,512742880,0d0d91c2-c9c2-4e81-90a5-86594dec0db9
6,2019-10-01 00:00:08 UTC,view,17300353,2053013553853497655,NaN,creed,380.96,555447699,4fe811e9-91de-46da-90c3-bbd87ed3a65d
7,2019-10-01 00:00:08 UTC,view,31500053,2053013558031024687,NaN,luminarc,41.16,550978835,6280d577-25c8-4147-99a7-abc6048498d6
8,2019-10-01 00:00:10 UTC,view,28719074,2053013565480109009,apparel.shoes.keds,baden,102.71,520571932,ac1cd4e5-a3ce-4224-a2d7-ff660a105880
9,2019-10-01 00:00:11 UTC,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb


In [4]:

print("=== INFORMACIÓN GENERAL ===")
df_sample.info()

print("\n=== TIPOS DE EVENTOS ===")
print(df_sample["event_type"].value_counts())

print("\n=== VALORES NULOS ===")
display(
    df_sample.isnull().sum()
    .to_frame("valores_nulos")
)

print("\n=== REGISTROS DUPLICADOS EXACTOS ===")
print(df_sample.duplicated().sum())


=== INFORMACIÓN GENERAL ===
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     100000 non-null  str    
 1   event_type     100000 non-null  str    
 2   product_id     100000 non-null  int64  
 3   category_id    100000 non-null  int64  
 4   category_code  67413 non-null   str    
 5   brand          85607 non-null   str    
 6   price          100000 non-null  float64
 7   user_id        100000 non-null  int64  
 8   user_session   100000 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 14.9 MB

=== TIPOS DE EVENTOS ===
event_type
view        97130
purchase     1655
cart         1215
Name: count, dtype: int64

=== VALORES NULOS ===


,valores_nulos
event_time,0
event_type,0
product_id,0
category_id,0
category_code,32587
brand,14393
price,0
user_id,0
user_session,0



=== REGISTROS DUPLICADOS EXACTOS ===
17


## Analizar usuarios, productos y sesiones

In [5]:

# Análisis de entidades y relaciones del dataset

print("=== ENTIDADES ÚNICAS ===")
print(f"Usuarios únicos: {df_sample['user_id'].nunique():,}")
print(f"Productos únicos: {df_sample['product_id'].nunique():,}")
print(f"Sesiones únicas: {df_sample['user_session'].nunique():,}")
print(f"Categorías únicas: {df_sample['category_id'].nunique():,}")

print("\n=== EVENTOS POR USUARIO ===")
eventos_usuario = df_sample.groupby("user_id").size()

print(f"Promedio: {eventos_usuario.mean():.2f}")
print(f"Mediana: {eventos_usuario.median():.2f}")
print(f"Máximo: {eventos_usuario.max():,}")

print("\n=== PRODUCTOS MÁS VISUALIZADOS ===")
top_views = (
    df_sample[df_sample["event_type"] == "view"]
    .groupby("product_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

display(top_views.to_frame("visualizaciones"))

print("\n=== SESIONES CON MÚLTIPLES PRODUCTOS COMPRADOS ===")

compras = df_sample[df_sample["event_type"] == "purchase"]

productos_por_sesion = (
    compras.groupby("user_session")["product_id"]
    .nunique()
)

sesiones_multiproducto = (productos_por_sesion >= 2).sum()

print(f"Total de sesiones con compras: {len(productos_por_sesion):,}")
print(f"Sesiones con 2+ productos distintos: {sesiones_multiproducto:,}")
print(f"Máximo de productos distintos: {productos_por_sesion.max() if len(productos_por_sesion) else 0}")


=== ENTIDADES ÚNICAS ===
Usuarios únicos: 20,384
Productos únicos: 20,621
Sesiones únicas: 24,382
Categorías únicas: 509

=== EVENTOS POR USUARIO ===
Promedio: 4.91
Mediana: 3.00
Máximo: 130

=== PRODUCTOS MÁS VISUALIZADOS ===


,visualizaciones
product_id,
1004856,994
1005115,920
1004767,793
1005105,547
1004870,546
1004833,455
1004249,448
5100816,421
1002544,395



=== SESIONES CON MÚLTIPLES PRODUCTOS COMPRADOS ===
Total de sesiones con compras: 1,431
Sesiones con 2+ productos distintos: 86
Máximo de productos distintos: 7


## Definir el esquema de eventos del pipeline

### 4. Normalización inicial de eventos

In [6]:

import pandas as pd
import numpy as np

# Copia de trabajo: preservamos la muestra original
df_events = df_sample.copy()

# ID estable para las filas originales de esta muestra
df_events.insert(
    0,
    "event_id",
    np.arange(len(df_events), dtype=np.int64)
)

# Conversión de timestamp a UTC
df_events["event_timestamp"] = pd.to_datetime(
    df_events["event_time"],
    utc=True,
    errors="coerce"
)

# Validación inicial
print("=== VALIDACIÓN ===")
print("Eventos:", len(df_events))
print("IDs únicos:", df_events["event_id"].is_unique)
print(
    "Timestamps inválidos:",
    df_events["event_timestamp"].isna().sum()
)

print("\n=== TIPOS DE DATOS ===")
print(df_events.dtypes)

print("\n=== MUESTRA NORMALIZADA ===")
display(df_events.head(5))


=== VALIDACIÓN ===
Eventos: 100000
IDs únicos: True
Timestamps inválidos: 0

=== TIPOS DE DATOS ===
event_id                         int64
event_time                         str
event_type                         str
product_id                       int64
category_id                      int64
category_code                      str
brand                              str
price                          float64
user_id                          int64
user_session                       str
event_timestamp    datetime64[us, UTC]
dtype: object

=== MUESTRA NORMALIZADA ===


,event_id,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_timestamp
0,0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c,2019-10-01 00:00:00+00:00
1,1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc,2019-10-01 00:00:00+00:00
2,2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8,2019-10-01 00:00:01+00:00
3,3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,2019-10-01 00:00:01+00:00
4,4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,2019-10-01 00:00:04+00:00


# Etapa 0. Ingesta y buffer FIFO

## 5. Etapa 0: Prueba de ingesta FIFO

In [7]:

import sys

# Permitir importar módulos de la carpeta src
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.ingestion import EventBufferFIFO

# Crear buffer
buffer = EventBufferFIFO()

# Seleccionar los primeros 5 eventos reales
eventos_prueba = df_events.head(5).to_dict("records")

# Encolar eventos
for evento in eventos_prueba:
    buffer.enqueue(evento)

print("=== INGESTA ===")
print("Eventos recibidos:", buffer.total_received)
print("Eventos pendientes:", len(buffer))

# Procesar eventos
print("\n=== PROCESAMIENTO FIFO ===")

while not buffer.is_empty():
    evento = buffer.dequeue()

    print(
        f"Evento {evento['event_id']} | "
        f"Tipo: {evento['event_type']} | "
        f"Producto: {evento['product_id']}"
    )

print("\n=== RESULTADOS ===")
print("Eventos procesados:", buffer.total_processed)
print("Eventos pendientes:", len(buffer))


=== INGESTA ===
Eventos recibidos: 5
Eventos pendientes: 5

=== PROCESAMIENTO FIFO ===
Evento 0 | Tipo: view | Producto: 44600062
Evento 1 | Tipo: view | Producto: 3900821
Evento 2 | Tipo: view | Producto: 17200506
Evento 3 | Tipo: view | Producto: 1307067
Evento 4 | Tipo: view | Producto: 1004237

=== RESULTADOS ===
Eventos procesados: 5
Eventos pendientes: 0


## Procesamiento FIFO con 100,000 eventos

In [9]:

import time
import tracemalloc

# Preparar eventos
eventos = df_events.to_dict("records")

# Crear una nueva cola
buffer_100k = EventBufferFIFO()

# Iniciar medición
tracemalloc.start()
inicio = time.perf_counter()

# Ingesta
for evento in eventos:
    buffer_100k.enqueue(evento)

tiempo_ingesta = time.perf_counter() - inicio

# Procesamiento
inicio_proceso = time.perf_counter()

primer_id = None
ultimo_id = None

while not buffer_100k.is_empty():
    evento = buffer_100k.dequeue()

    if primer_id is None:
        primer_id = evento["event_id"]

    ultimo_id = evento["event_id"]

tiempo_proceso = time.perf_counter() - inicio_proceso

memoria_actual, memoria_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("=== RESULTADOS FIFO 100K ===")
print(f"Eventos recibidos: {buffer_100k.total_received:,}")
print(f"Eventos procesados: {buffer_100k.total_processed:,}")
print(f"Pendientes: {len(buffer_100k):,}")

print(f"\nPrimer ID: {primer_id}")
print(f"Último ID: {ultimo_id}")

print(f"\nTiempo de ingesta: {tiempo_ingesta:.4f} s")
print(f"Tiempo de procesamiento: {tiempo_proceso:.4f} s")
print(f"Tiempo total: {tiempo_ingesta + tiempo_proceso:.4f} s")

print(f"\nMemoria pico medida: {memoria_pico / 1024**2:.2f} MiB")


=== RESULTADOS FIFO 100K ===
Eventos recibidos: 100,000
Eventos procesados: 100,000
Pendientes: 0

Primer ID: 0
Último ID: 99999

Tiempo de ingesta: 0.0494 s
Tiempo de procesamiento: 0.0930 s
Tiempo total: 0.1424 s

Memoria pico medida: 0.81 MiB


## 7. Etapa 1: Preprocesamiento y deduplicación

In [10]:

from src.preprocessing import preprocess_batch

# Tomamos 10 eventos reales de REES46
eventos_originales = df_events.head(10).to_dict("records")

# Simulamos dos retransmisiones del mismo evento
eventos_prueba = (
    eventos_originales
    + [eventos_originales[2].copy()]
    + [eventos_originales[5].copy()]
)

print("=== ANTES DEL PREPROCESAMIENTO ===")
print("Eventos recibidos:", len(eventos_prueba))

# Ejecutamos la Etapa 1
resultado = preprocess_batch(eventos_prueba)

print("\n=== DESPUÉS DEL PREPROCESAMIENTO ===")
print("Eventos válidos:", resultado["valid"])
print("Eventos inválidos:", resultado["invalid"])
print("Duplicados técnicos:", resultado["duplicates"])
print("Eventos finales:", len(resultado["events"]))

print("\n=== IDs PROCESADOS ===")
print([event["event_id"] for event in resultado["events"]])


=== ANTES DEL PREPROCESAMIENTO ===
Eventos recibidos: 12

=== DESPUÉS DEL PREPROCESAMIENTO ===
Eventos válidos: 12
Eventos inválidos: 0
Duplicados técnicos: 2
Eventos finales: 10

=== IDs PROCESADOS ===
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 8. Integración de etapas 0 y 1

In [11]:

from src.pipeline import run_ingestion_pipeline

# Utilizamos los 100 000 eventos normalizados
eventos_pipeline = df_events.to_dict("records")

resultado_pipeline = run_ingestion_pipeline(
    events=eventos_pipeline,
    batch_size=5000
)

print("=== PIPELINE ETAPAS 0 Y 1 ===")

for clave, valor in resultado_pipeline["metrics"].items():
    print(f"{clave}: {valor:,}")

print("\n=== VERIFICACIONES ===")

ids = [
    evento["event_id"]
    for evento in resultado_pipeline["events"]
]

print("Total de IDs:", len(ids))
print("IDs únicos:", len(ids) == len(set(ids)))
print("Primer ID:", ids[0])
print("Último ID:", ids[-1])


=== PIPELINE ETAPAS 0 Y 1 ===
received: 100,000
valid: 100,000
invalid: 0
duplicates: 0
output: 100,000

=== VERIFICACIONES ===
Total de IDs: 100000
IDs únicos: True
Primer ID: 0
Último ID: 99999
